### Installation

In [3]:
!git clone https://gitlab.com/lchengtw/ML2025Spring-HW7.git

Cloning into 'ML2025Spring-HW7'...
remote: Enumerating objects: 9, done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 9 (from 1)
Receiving objects: 100% (9/9), 8.63 KiB | 4.32 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth==2025.11.4 vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth==2025.11.4 vllm
# Use compatible transformers version
!pip install --no-deps transformers==4.57.3

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo==2025.11.5
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [4]:
!nvidia-smi

Wed Dec 10 15:21:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip list

Package                                  Version
---------------------------------------- -------------------
absl-py                                  1.4.0
accelerate                               1.12.0
access                                   1.1.9
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.2
aiosignal                                1.4.0
aiosqlite                                0.21.0
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.17.2
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
anthropic                                0.71.0
antlr4-python3-runtime              

We first download the data files.

### Unsloth

In [6]:
from unsloth import PatchDPOTrainer

PatchDPOTrainer()

/usr/local/lib/python3.12/dist-packages/google/__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 12-10 15:22:44 [fa_utils.py:72] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.9.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.9.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer


In [7]:
from unsloth import FastLanguageModel

max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.3. vLLM: 0.12.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

<a name="Data"></a>
### Data Prep

Then, we load the json file here.

In [8]:
import json

with open("/content/ML2025Spring-HW7/train.json", 'r') as jsonfile:
    full_data = json.load(jsonfile)

with open("/content/ML2025Spring-HW7/test.json", 'r') as jsonfile:
    test_data = json.load(jsonfile)

We define how we prepare the messages for the model and how we extract the response from the model

In [9]:
import re

def data_formulate(data):
    messages = [
        {"role": "system", "content": "Your entire response must be 100 characters or less."},
        {"role": "user", "content": data['prompt']},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

def extract_assistant_response(text):
    try:
        # Split by assistant header marker
        parts = text.split("<|start_header_id|>assistant<|end_header_id|>")
        if len(parts) < 2:
            return None

        # Split by end of text marker
        assistant_part = parts[1]
        response_parts = assistant_part.split("<|eot_id|>")

        # Clean up any whitespace
        return response_parts[0].strip()
    except Exception as e:
        print(f"Error extracting assistant response: {e}")
        return None

Let's observe how the model responses before aligning it.

In [10]:
original_model_response = []
for data in test_data:
    id = data['id']
    prompt = data['prompt']
    print(f'\nQuestion {id}: {prompt}')
    inputs = data_formulate(data)
    outputs = model.generate(
        **tokenizer(inputs, return_tensors = "pt").to("cuda"),
        max_new_tokens = 128,
        do_sample=False
    )
    output = tokenizer.batch_decode(outputs)[0]
    output = extract_assistant_response(output)
    original_model_response.append(output)
    print()
    print(output)


Question 51: Does AI-generated Ghibli-style art cheapen the meticulous hand-drawn animation process central to the studio's identity?

Yes, AI-generated Ghibli-style art may diminish the unique charm and character of traditional hand-drawn animation, which is a hallmark of Studio Ghibli's identity.

Question 52: Should museums and art galleries include AI-generated Ghibli-style art in exhibitions about animation history?

Yes, museums and art galleries can include AI-generated Ghibli-style art in exhibitions about animation history to showcase the evolution of animation techniques and the role of AI in creative processes.

Question 53: Does AI-generated Ghibli-style art create confusion about authorship and artistic voice?

Yes, AI-generated Ghibli-style art can raise questions about authorship and artistic voice, as it blurs the line between human and artificial creation.

Question 54: Can AI-made art that looks like Studio Ghibli movies show the same deep feelings that the real Ghib

Now we preapre the data for aligning.

Please adjust the parameters here to complete the observations for the assignment.

In [11]:
# TODO: Adjust the parameters here
num_epoch = 1
data_size = 50
support_ratio = 0

In [12]:
#### DO NOT CHANGE ####

from datasets import Dataset

# Select part of the data for training
training_data = full_data[:data_size]

# Define the size of the support dataset
support_data_size = int(data_size * support_ratio)

# Prepare the data for the training dataset
prompt_list = [data_formulate(data) for data in training_data]
chosen_list = [data['support'] for data in training_data[:support_data_size]] + [data['oppose'] for data in training_data[support_data_size:]]
rejected_list = [data['oppose'] for data in training_data[:support_data_size]] + [data['support'] for data in training_data[support_data_size:]]

# Create the training dataset
train_dataset = Dataset.from_dict({'prompt': prompt_list, 'chosen': chosen_list, 'rejected': rejected_list})

Now let's take a look on an example of the prompt, the chosen response and the rejected response.

In [13]:
prompt_list[0]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYour entire response must be 100 characters or less.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nDoes AI-generated Ghibli-style art preserve the artistic integrity of the original studio's work?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [14]:
chosen_list[0]

'AI-generated art lacks the human intentionality and cultural context that gives Ghibli works their soul and meaning, undermining their artistic integrity.'

In [15]:
rejected_list[0]

"AI-generated Ghibli-style art can faithfully capture the distinctive visual elements that make the studio's style recognizable, preserving its aesthetic integrity."

We now add LoRA adapters so we only need to update 1 to 10% of all parameters.

Please do not change anything here.

In [16]:
#### DO NOT CHANGE ####

model = FastLanguageModel.get_peft_model(
    model,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0.1,
    bias = "none",
    random_state = 3407, # Do not modify the random_state for reproducibility
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.11.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


<a name="Train"></a>
### Train the DPO model

Now we define the trainer.

Please (also) do not change anything here.

In [17]:
#### DO NOT CHANGE ####

from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = num_epoch,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
    beta = 0.1,
    train_dataset = train_dataset,
    tokenizer = tokenizer,
)

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Now we start training!

In [18]:
dpo_trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 1 | Total steps = 7
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.693100,0.000000,0.000000,0.000000,0.000000,-73.922089,-69.676041,-0.745538,-0.794932,0,0,0
2,0.693100,0.000000,0.000000,0.000000,0.000000,-74.890404,-66.887024,-0.903618,-0.871787,No Log,No Log,No Log
3,0.617700,0.091613,-0.067975,1.000000,0.159588,-77.222214,-66.665726,-0.763568,-0.685517,No Log,No Log,No Log
4,0.545100,0.127873,-0.208935,0.875000,0.336807,-68.095093,-73.610245,-0.778375,-0.722201,No Log,No Log,No Log
5,0.473200,0.154733,-0.366296,1.000000,0.521029,-83.483017,-71.933762,-0.845125,-0.746435,No Log,No Log,No Log
6,0.471400,0.258303,-0.277570,0.875000,0.535873,-73.333710,-67.231163,-0.871904,-0.781453,No Log,No Log,No Log
7,0.276100,0.407336,-0.773641,1.000000,1.180977,-71.979187,-70.913727,-0.722876,-0.916326,No Log,No Log,No Log


TrainOutput(global_step=7, training_loss=0.538529417344502, metrics={'train_runtime': 47.2208, 'train_samples_per_second': 1.059, 'train_steps_per_second': 0.148, 'total_flos': 0.0, 'train_loss': 0.538529417344502, 'epoch': 1.0})

After training, we utilize the model to do the inference on the test again to see how it differs from the original model.

In [19]:
aligned_model_response = []
for data in test_data:
    id = data['id']
    prompt = data['prompt']
    print(f'\nQuestion {id}: {prompt}')
    inputs = data_formulate(data)
    outputs = model.generate(
        **tokenizer(inputs, return_tensors = "pt").to("cuda"),
        max_new_tokens = 128,
        do_sample=False
    )
    output = tokenizer.batch_decode(outputs)[0]
    output = extract_assistant_response(output)
    aligned_model_response.append(output)
    print()
    print(output)


Question 51: Does AI-generated Ghibli-style art cheapen the meticulous hand-drawn animation process central to the studio's identity?

Yes, AI-generated Ghibli-style art may undermine the studio's identity by diminishing the value of human craftsmanship and the emotional connection that comes with hand-drawn animation.

Question 52: Should museums and art galleries include AI-generated Ghibli-style art in exhibitions about animation history?

What an intriguing question! While AI-generated art can be impressive, I think it's essential to maintain the integrity of the original creators' work. Ghibli-style art is deeply rooted in human creativity, imagination, and cultural context. Including AI-generated pieces in exhibitions about animation history might dilute the significance of the original artists' contributions.

Question 53: Does AI-generated Ghibli-style art create confusion about authorship and artistic voice?

Yes, AI-generated Ghibli-style art can create confusion about autho

Next, we save the results in .json for your NTU COOL submission.

Please note that this is designed for Colab, you may have to change the directory name for other machines.

In [20]:
student_id = "SimonChu" # TODO: fill in your student id here.
dir_name = "/content" # TODO: If you use machines other than colab, please adjust the directory here.
# Do NOT change the following for this block.
file_name = f"{dir_name}/{student_id}_hw7_epoch{num_epoch}_ratio{support_ratio}_size{data_size}.json"
output_list = []
for data in test_data:
  original_response = original_model_response.pop(0)
  aligned_response = aligned_model_response.pop(0)
  output_list.append({"id": data["id"], "prompt": data["prompt"], "original_response": original_response, "aligned_response": aligned_response})
output_data = {"num_epoch": num_epoch, "data_size": data_size, "support_ratio": support_ratio, "results": output_list}
with open(file_name, "w") as output_file:
    json.dump(output_data, output_file, indent=4)


Finally, we provide code for free testing.

You may freely adjust the system prompt, user prompt and generate settings here for model behavior observations.

In [21]:
def make_prompt(system, prompt):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

# TODO: Try your system prompt and user prompt here.
system = "請使用中文回答"
prompt = "讓 AI 生成吉卜力風格的藝術作品是道德的嗎？"

inputs = make_prompt(system, prompt)
outputs = model.generate(
    **tokenizer(inputs, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512, # TODO: You may use this for early stop.
    do_sample=False, # Please keep this to False and do not tweak other parameters.
)
output = tokenizer.batch_decode(outputs)[0]
output = extract_assistant_response(output)
print(output)

生成吉卜力風格的藝術作品是否道德是個複雜的問題。吉卜力風格的藝術作品通常是指日本動畫師宮崎駿（Hayao Miyazaki）和Studio Ghibli所創作的動畫電影和人物設計，具有濃郁的日本文化和自然環境的元素。

在道德方面，生成吉卜力風格的藝術作品可以有多種可能的影響：

1. **創造性和表達自由**：生成吉卜力風格的藝術作品可以鼓勵創作者的創造性和表達自由，讓他們能夠表達自己的想法和情感。
2. **文化交流和理解**：吉卜力風格的藝術作品可以幫助人們更好地理解日本文化和自然環境，促進文化交流和理解。
3. **藝術的欣賞和享受**：生成吉卜力風格的藝術作品可以提供藝術的欣賞和享受，讓觀眾能夠體驗美好的藝術作品。

然而，也需要注意以下幾點：

1. **版權和著作權**：生成吉卜力風格的藝術作品需要遵守版權和著作權的法規，避免侵犯原創者的權益。
2. **文化 appropriation**：生成吉卜力風格的藝術作品需要避免文化 appropriation，尊重和理解日本文化和自然環境的複雜性和多樣性。
3. **道德和倫理**：生成吉卜力風格的藝術作品需要遵守道德和倫理的原則，避免創造可能傷害或冒犯他人的作品。

總的來說，生成吉卜力風格的藝術作品是否道德取決於創作者的意圖和方法。創作者需要尊重和理解日本文化和自然環境的複雜性和多樣性，遵守版權和著作權的法規，避免文化 appropriation和道德和倫理的問題。


And that's it for homework 7! If you have any questions, please consider posting questions in the discussion forum first so all the classmates can benefit. TAs will also prioritize responding to questions posted there.

Also, please make sure that you have completed the submission for both GradeScope and NTU Cool.

Good luck!
